# CFD-Drag-2D: flow-field dataset generation (M7, v2)

Same as the M4 notebook (same 5000 cases, seed 0), but the solver also time-averages the flow
field and the dataset stores a 256 x 96 window around each obstacle (`fields.npy`, float16:
ux / U, uy / U, Cp). Cd and Cl must equal dataset v1.
Output (`/kaggle/working/cfd_drag_2d_fields`): `data.csv`, `masks.npy`, `fields.npy`, `config.json`,
`summary.json`, `spot_check.json`, `run_info.json`, `README.md`.

In [ ]:
REPO = "https://github.com/PradeepkrishnaGM/CFD-Drag-2D.git"
COMMIT = "main"   # branch or commit SHA; the resolved SHA is recorded in run_info.json
N = 5000          # cases (indices 0..N-1)
SEED = 0
SPOT_K = 8        # cases re-solved on the CPU
OUT = "/kaggle/working/cfd_drag_2d_fields"
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!rm -rf /tmp/repo && git clone -q $REPO /tmp/repo && git -C /tmp/repo checkout -q $COMMIT && git -C /tmp/repo log --oneline -1

In [ ]:
%cd /tmp/repo
!pip install -q -e . --no-deps && pip install -q pytest
import os, subprocess
nvcc_dir = subprocess.run(["bash", "scripts/kaggle_setup_cuda.sh"], capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
os.environ["PATH"] = nvcc_dir + ":" + os.environ["PATH"]
!nvcc --version | tail -2
!make -s solver && make -s cuda && ls -la build

In [ ]:
# the GPU tests compare CUDA and CPU results, including the mean fields; stop on any failure
r = subprocess.run(["python", "-m", "pytest", "-q", "-m", "not slow", "-rs"], capture_output=True, text=True)
print(r.stdout[-3000:])
assert r.returncode == 0, "tests failed"

In [ ]:
import json, time
t0 = time.time()
!python scripts/generate.py --backend cuda --n $N --seed $SEED --fields --out $OUT
gen_seconds = time.time() - t0
print(f"generation: {gen_seconds:.0f} s, {gen_seconds / N:.2f} s/case")

In [ ]:
from cfd_drag.check import spot_check
spot = spot_check(OUT, k=SPOT_K, workers=os.cpu_count())
json.dump(spot, open(f"{OUT}/spot_check.json", "w"), indent=2)
print(json.dumps(spot, indent=1))
assert spot["passed"], "GPU rows differ from CPU re-runs"

In [ ]:
!python scripts/check_dataset.py --data $OUT

In [ ]:
import platform
summary = json.load(open(f"{OUT}/summary.json"))
config = json.load(open(f"{OUT}/config.json"))
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.split("\n")
run_info = {"commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
            "n": N, "seed": SEED, "gpus": [g for g in gpus if g], "generation_seconds": round(gen_seconds, 1),
            "seconds_per_case": round(gen_seconds / N, 3), "python": platform.python_version(),
            "date_utc": time.strftime("%Y-%m-%d %H:%M", time.gmtime())}
json.dump(run_info, open(f"{OUT}/run_info.json", "w"), indent=2)

q = lambda d: f"{d['min']:.3g} / {d['median']:.3g} / {d['max']:.3g}"
card = f"""# CFD-Drag-2D: 2D obstacle flow-field dataset (lattice Boltzmann)

{summary['n']} random 2D obstacles (circle, rectangle, triangle) in a channel, each solved with a
validated lattice Boltzmann solver (TRT collision, D2Q9), with the time-averaged flow field around
each obstacle. Made for training surrogates that predict the flow field (U-Net) or drag and lift
(CNN) from the obstacle mask and the Reynolds number. Same cases as dataset v1 (Cd, Cl identical).
Code: https://github.com/PradeepkrishnaGM/CFD-Drag-2D (commit `{run_info['commit'][:10]}`).

## Files
- `data.csv`: one row per case; inputs (shape, size1, size2, angle, re, D, blockage) and results
  (tau, Cd, Cl, Cl_rms, cd_drift, pressure_loss, status, ok, error). Use rows with `ok == True`.
- `masks.npy`: uint8 array ({', '.join(map(str, [summary['n']] + summary['mask_shape']))}), 1 = solid, cropped around the obstacle; row i = case i.
- `fields.npy`: float16 array {tuple(summary['fields']['shape'])}: time-averaged ux / U, uy / U and
  Cp = (p - p_outlet) / (0.5 U^2) in a {config['fields']['width']} x {config['fields']['height']} window (x from
  {config['fields']['x_before']} cells upstream of the obstacle centre; wall rows excluded); zeros for failed rows.
  The obstacle mask in that window: `cfd_drag.dataset.window_mask(masks[i], ...)`.
- `config.json`: every solver and sampling parameter. `summary.json`: checks and statistics.
- `spot_check.json`: {len(spot['cases'])} cases re-solved on the CPU; they match the GPU rows (relative tolerance {spot['rel_tol']}).
- `run_info.json`: commit, GPUs, timing.

## Contents
- Cases solved cleanly: {summary['n_ok']} of {summary['n']} ({summary['ok_fraction']:.1%})
- Shapes: {summary['shapes']}; flow regimes: {summary['regimes']}
- Re (min / median / max): {q(summary['inputs']['re'])}; frontal height D [cells]: {q(summary['inputs']['D'])}
- Cd: {q(summary['targets']['Cd'])}; Cl: {q(summary['targets']['Cl'])}
- Field checks passed: no-slip on obstacle cells, volume flux constant along the window
  (max deviation {summary['fields']['max_flux_deviation']:.2%})

## Setup
Channel {config['domain']['nx']} x {config['domain']['ny']} cells, no-slip walls, inlet velocity
{config['physics']['u']} (lattice units), obstacle centre at x = nx/4. Cd, Cl and the fields are time-averaged
over {config['physics']['avg_flowthroughs']} flow-through after {config['physics']['warmup_flowthroughs']} flow-throughs of warm-up,
using the projected frontal height D as reference length. Blockage D/H is capped at
{config['physics']['max_blockage']}, so Cd is higher than in free stream.
Seed {SEED}; the dataset is reproducible from the code at the commit above.

License: MIT.
"""
open(f"{OUT}/README.md", "w").write(card)
print(card)

In [ ]:
# keep only the small consolidated files
!rm -rf $OUT/samples && ls -la $OUT && du -sh $OUT